<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Indian Metropolitian City Rent Prediction </b></div>

* This study focuses on conducting regression analysis using Rent price in Indian top tier cities
* This study begins with simpler regression algorithms to establish a baseline, followed by boosting methods such as GradientBoosting and XGBoost. Finally, hyperparameter tuning using the OPTUNA framework will be applied to select the best-performing algorithm and if we overfit we shift to Voting/Bagging models.
* The objective is to maximize the performance metric and outline the stages of machine learning for the regression task.


#### Some Information about the dataset
##### India's housing landscape is diverse, spanning from ancient palaces to modern high-rises and rural huts. Housing growth parallels rising incomes. Renting, integral to the sharing economy, offers flexible living solutions. The dataset covers 4700+ properties with details like BHK, rent, size, floors, area type, city, furnishing, tenant preferences, bathrooms, and contact info so we try to predict its Rate using the best possible model.

##### Information about each feature:
- BHK: Number of Bedrooms, Hall, Kitchen.
- Rent: Rent of the Houses/Apartments/Flats.
- Size: Size of the Houses/Apartments/Flats in Square Feet.
- Floor: Houses/Apartments/Flats situated in which Floor and Total Number of Floors (Example: Ground out of 2, 3 out of 5, etc.)
- Area Type: Size of the Houses/Apartments/Flats calculated on either Super Area or Carpet Area or Build Area.
- Area Locality: Locality of the Houses/Apartments/Flats.
- City: City where the Houses/Apartments/Flats are Located.
- Furnishing Status: Furnishing Status of the Houses/Apartments/Flats, either it is Furnished or Semi-Furnished or Unfurnished.
- Tenant Preferred: Type of Tenant Preferred by the Owner or Agent.
- Bathroom: Number of Bathrooms.
- Point of Contact: Whom should you contact for more information regarding the Houses/Apartments/Flats.

In [2]:
import pandas as pd
df = pd.read_csv('/kaggle/input/house-rent-prediction-dataset/House_Rent_Dataset.csv')
df.head()

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


### Our approach strategy includes:
- Exploring each feature thoroughly.
- Assessing the correlation of each feature with the target variable.
- Performing scaling, imputation, and outlier removal as needed.
- Handling categorical features appropriately.
- Identifying patterns and relationships within the data.
- Conducting feature selection to enhance model performance.
- Evaluating various models to determine the optimal choice.
- Tuning the hyperparameters of the selected model using optimization algorithms.
- Considering ensemble methods like bagging or voting to mitigate overfitting if necessary, as they generally provide robust performance.

#### Let us look at the Dtype for each column 

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


In [4]:
print(f'DataFrame size: {df.shape}')

DataFrame size: (4746, 12)


#### The below code is just to seperate Current floor and Total Floor, because by intuition we know that in India most of flats are cheaper in lower floors so It will be useful later on and also it is object Data type so anyways we need to convert it to numerical column lol

In [5]:
# Regular expression to extract the current floor and total number of floors
pattern = r'(?P<CurrentFloor>\w+)\s*out\s*of\s*(?P<TotalFloors>\d+)'

# Extract the values into new columns
df[['CurrentFloor', 'TotalFloors']] = df['Floor'].str.extract(pattern)

# Replace non-numeric floor labels with specific values
floor_replacements = {
    'Ground': 0,
    'Basement': -1  # Assuming 'Basement' should be treated as floor -1
}

df['CurrentFloor'] = df['CurrentFloor'].replace(floor_replacements)

# Convert columns to appropriate data types
df['CurrentFloor'] = pd.to_numeric(df['CurrentFloor'], errors='coerce')
df['TotalFloors'] = pd.to_numeric(df['TotalFloors'], errors='coerce')
df = df.drop(columns=['Floor'])
df.head()

,Posted On,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors
0,2022-05-18,2,10000,1100,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner,0.0,2.0
1,2022-05-13,2,20000,800,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0
2,2022-05-16,2,17000,1000,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0
3,2022-07-04,2,10000,800,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner,1.0,2.0
4,2022-05-09,2,7500,850,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner,1.0,2.0


#### Now we will check the number of categorical feature with number of unique value

In [6]:
categorical_columns = df.select_dtypes(include=['object']).columns
unique_counts = df[categorical_columns].nunique()
print(unique_counts)

Posted On              81
Area Type               3
Area Locality        2235
City                    6
Furnishing Status       3
Tenant Preferred        3
Point of Contact        3
dtype: int64


#### Observation: Out of 4746 columns we have 2235 which is nearabout 50% so I dont think Area Type column will contribute much to our model due to High Cardinality but anyways we will get feature importance through various techniques later on

In [7]:
print(f'Number of missing values in DataFrame:\n{df.isna().sum()}')

Number of missing values in DataFrame:
Posted On            0
BHK                  0
Rent                 0
Size                 0
Area Type            0
Area Locality        0
City                 0
Furnishing Status    0
Tenant Preferred     0
Bathroom             0
Point of Contact     0
CurrentFloor         4
TotalFloors          4
dtype: int64


In [8]:
# We will use KNN Imputer to fill these missing values, Though we can drop it too but anyways even if we get it as outlier we can remove it later on
from sklearn.impute import KNNImputer
columns_for_imputation = ['CurrentFloor', 'TotalFloors']
imputer = KNNImputer(n_neighbors=3)
df[columns_for_imputation] = imputer.fit_transform(df[columns_for_imputation])

In [9]:
print(f'Number of missing values in DataFrame:\n{df.isna().sum()}')

Number of missing values in DataFrame:
Posted On            0
BHK                  0
Rent                 0
Size                 0
Area Type            0
Area Locality        0
City                 0
Furnishing Status    0
Tenant Preferred     0
Bathroom             0
Point of Contact     0
CurrentFloor         0
TotalFloors          0
dtype: int64


#### We will seperate out the Year from Posted On column because price increases in a year due to inflation or any reason so finding out pattern day or month wise is difficult

In [10]:
df['Year'] = pd.to_datetime(df['Posted On']).dt.year
df = df.drop(columns=['Posted On'])
df.head()

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors,Year
0,2,10000,1100,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner,0.0,2.0,2022
1,2,20000,800,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0,2022
2,2,17000,1000,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0,2022
3,2,10000,800,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner,1.0,2.0,2022
4,2,7500,850,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner,1.0,2.0,2022


In [11]:
value_counts = df['City'].value_counts()
print(value_counts)

City
Mumbai       972
Chennai      891
Bangalore    886
Hyderabad    868
Delhi        605
Kolkata      524
Name: count, dtype: int64


#### Observation: Almost equally distributed so it is balanced which is actually desired

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneak Peek into the data 👀 </b></div>

#### Inporting all Important Libraries we gonna use

In [12]:
import numpy as np  
import pandas as pd  
import matplotlib.pyplot as plt  
import seaborn as sns  
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import optuna
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from xgboost import XGBRegressor
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')  # ignore notifications
import sys
import os

#### Finding out some basic pattern using sunburst,the sunburst chart below visually represents the distribution of flats based on various attributes: City, Area Type, Furnishing Status, and Tenant Preferred. It provides an intuitive way to explore how different combinations of these factors contribute to the overall allocation of flats, highlighting patterns and relationships within the dataset.

In [13]:
fig = px.sunburst(
    df, 
    path=['City', 'Area Type', 'Furnishing Status', 'Tenant Preferred'], 
    width=700,
    height=700,
    title='Allotment of flats according to Bachelors/Family/(Bachelors/Family)',
    color_discrete_sequence=px.colors.cyclical.Phase
)

fig.show()

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneak Peek into Boxplot 👀 </b></div>

In [55]:
scl = [ [0,"rgb(5, 10, 172)"],[0.35,"rgb(40, 60, 190)"],[0.5,"rgb(70, 100, 245)"],\
    [0.6,"rgb(90, 120, 245)"],[0.7,"rgb(106, 137, 247)"],[1,"rgb(220, 220, 220)"] ]
fig = px.box(df, x="Furnishing Status", y="Rent", template='plotly_dark', title='<b>Boxplots', color='Furnishing Status')
fig.update_layout(title_x=0.5)
fig.show()

#### Observation: Unfurnished houses for rent are having less rent as compared to others.

In [56]:
scl = [ [0,"rgb(5, 10, 172)"],[0.35,"rgb(40, 60, 190)"],[0.5,"rgb(70, 100, 245)"],\
    [0.6,"rgb(90, 120, 245)"],[0.7,"rgb(106, 137, 247)"],[1,"rgb(220, 220, 220)"] ]
fig = px.box(df, x="Point of Contact", y="Rent", template='plotly_dark', title='<b>Boxplots', color='Point of Contact')
fig.update_layout(title_x=0.5)
fig.show()

#### Observations: The higher rent for properties obtained through a contact agent may reflect additional costs to compensate the agent's services. In contrast, renting directly from builders tends to be less expensive.

In [ ]:
scl = [ [0,"rgb(5, 10, 172)"],[0.35,"rgb(40, 60, 190)"],[0.5,"rgb(70, 100, 245)"],\
    [0.6,"rgb(90, 120, 245)"],[0.7,"rgb(106, 137, 247)"],[1,"rgb(220, 220, 220)"] ]
fig = px.box(df, x="Tenant Preferred", y="Rent", template='plotly_dark', title='<b>Boxplots', color='Tenant Preferred')
fig.update_layout(title_x=0.5)
fig.show()

#### Objservation: Houses rented with no tenant conditions are having a bit more rent as compared to others.

In [ ]:
scl = [ [0,"rgb(5, 10, 172)"],[0.35,"rgb(40, 60, 190)"],[0.5,"rgb(70, 100, 245)"],\
    [0.6,"rgb(90, 120, 245)"],[0.7,"rgb(106, 137, 247)"],[1,"rgb(220, 220, 220)"] ]
fig = px.box(df, x="City", y="Rent", template='plotly_dark', title='<b>Boxplots', color='City')
fig.update_layout(title_x=0.5)
fig.show()

#### Observations: Mumbai has a high demand for housing, leading to higher rents, likely due to the high influx of job seekers and corporate relocations. Other cities, except Kolkata, have relatively equal rent levels. Kolkata has lower rents, which can be attributed to its comparatively less developed job sectors and lifestyle, resulting in lower demand for rental properties.

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Sneak Peek into each feature one by one 👀 </b></div>

In [ ]:
import plotly.express as px
grouped_df = df.groupby('City', as_index=False)['Rent'].mean()
sorted_df = grouped_df.sort_values(by='Rent', ascending=False)

fig = px.bar(sorted_df, x='City', y='Rent', color_discrete_sequence=['steelblue'], template='plotly_dark', title='<b> Average Rent by City </b>')
fig.update_layout(title_x=0.5)
fig.show()

#### Observation: As observed before Housess in mumbai have relatively high prices for so and so reason

In [ ]:
import plotly.express as px
grouped_df = df.groupby('Furnishing Status', as_index=False)['Rent'].mean()
sorted_df = grouped_df.sort_values(by='Rent', ascending=False)

fig = px.bar(sorted_df, x='Furnishing Status', y='Rent', color_discrete_sequence=['steelblue'], template='plotly_dark', title='<b> Average Rent by Furnishing Status </b>')
fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
import plotly.express as px
grouped_df = df.groupby('BHK', as_index=False)['Rent'].mean()
sorted_df = grouped_df.sort_values(by='Rent', ascending=False)

fig = px.bar(sorted_df, x='BHK', y='Rent', color_discrete_sequence=['steelblue'], template='plotly_dark', title='<b> Average Rent by Furnishing Status </b>')
fig.update_layout(title_x=0.5)
fig.show()

#### Observation: Rent prices of houses tend to increase with the number of bedrooms (BHK). This trend reflects larger properties commanding higher rents due to their increased space and amenities. Conversely, 6 BHK properties are less sought after, resulting in lower rental prices due to lower demand (Reason Unknown lol)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('<i>Rent', '<i>Size'))
fig.add_trace(go.Histogram(x=df['Rent'], name='Rent'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Size'], name='Size'), row=1, col=2)
fig.update_layout(template='plotly_dark', title_text='<b>Distributions', title_x=0.5)

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Linear dependency on Size and Rent </b></div>

In [ ]:
fig = px.scatter(df, x="Size", y="Rent", trendline="ols", color_discrete_sequence=['steelbl'],
                template='plotly_dark', title='<b>Target variable dependency on median_income', color='Rent', color_continuous_scale=px.colors.sequential.Blues)
fig.update_layout(title_x=0.5)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='Rent',
    y='Size',
    color='Size',
    size='Size',
    color_continuous_scale='viridis',
    title='Scatter plot of Rent vs Size',
    labels={'Rent': 'Rent Amount', 'Size': 'Size of the Apartment'},
    template='plotly_dark'
)

fig.update_layout(
    title_x=0.5,
    width=800,
    height=400
)
fig.show()

In [ ]:
categorical_columns = df.select_dtypes(include=['object']).columns
unique_counts = df[categorical_columns].nunique()
print(unique_counts)

Area Type               3
Area Locality        2235
City                    6
Furnishing Status       3
Tenant Preferred        3
Point of Contact        3
dtype: int64


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Converting Categorical to numerical using LabelEncoder </b></div>

In [14]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['Area Type'] = label_encoder.fit_transform(df['Area Type'])
df['Area Locality'] = label_encoder.fit_transform(df['Area Locality'])
df['City'] = label_encoder.fit_transform(df['City'])
df['Furnishing Status'] = label_encoder.fit_transform(df['Furnishing Status'])
df['Tenant Preferred'] = label_encoder.fit_transform(df['Tenant Preferred'])
df['Point of Contact'] = label_encoder.fit_transform(df['Point of Contact'])

df.head()

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors,Year
0,2,10000,1100,2,221,4,2,1,2,2,0.0,2.0,2022
1,2,20000,800,2,1527,4,1,1,1,2,1.0,3.0,2022
2,2,17000,1000,2,1760,4,1,1,1,2,1.0,3.0,2022
3,2,10000,800,2,526,4,2,1,1,2,1.0,2.0,2022
4,2,7500,850,1,1890,4,2,0,1,2,1.0,2.0,2022


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Feature correlation and Feature selection </b></div>

In [15]:
df.corr()['Rent']

BHK                  0.369718
Rent                 1.000000
Size                 0.413551
Area Type           -0.214867
Area Locality       -0.018849
City                 0.210525
Furnishing Status   -0.146251
Tenant Preferred     0.006027
Bathroom             0.441215
Point of Contact    -0.339677
CurrentFloor         0.326358
TotalFloors          0.352205
Year                      NaN
Name: Rent, dtype: float64

#### We can see already that Area type is the least dependent upon target variable (Rent)

In [68]:
fig=px.imshow(df.corr(),text_auto=True, template='plotly_dark', color_continuous_scale=px.colors.sequential.Blues, aspect='auto',title='<b>Correlation matrix')
fig.update_layout(title_x=0.5)
fig.show()

In [16]:
df.corr()['Area Type']

BHK                 -0.153225
Rent                -0.214867
Size                -0.079705
Area Type            1.000000
Area Locality       -0.007743
City                -0.282856
Furnishing Status    0.056276
Tenant Preferred     0.155388
Bathroom            -0.183012
Point of Contact     0.559451
CurrentFloor        -0.254626
TotalFloors         -0.285407
Year                      NaN
Name: Area Type, dtype: float64

In [17]:
value_counts = df['Year'].value_counts()
print(value_counts)

Year
2022    4746
Name: count, dtype: int64


#### Ah we did useless hardwork before as in this dataset we just have datas of 2022 lmao so let us just drop it

In [18]:
columns_to_drop = ['Year']
df = df.drop(columns=columns_to_drop)
df.head()

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors
0,2,10000,1100,2,221,4,2,1,2,2,0.0,2.0
1,2,20000,800,2,1527,4,1,1,1,2,1.0,3.0
2,2,17000,1000,2,1760,4,1,1,1,2,1.0,3.0
3,2,10000,800,2,526,4,2,1,1,2,1.0,2.0
4,2,7500,850,1,1890,4,2,0,1,2,1.0,2.0


In [19]:
# Removing Outliers
Q1 = df['Rent'].quantile(0.25)
Q3 = df['Rent'].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df['Rent'] < (Q1 - 1.5 * IQR)) | (df['Rent'] > (Q3 + 1.5 * IQR)))]
df

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors
0,2,10000,1100,2,221,4,2,1,2,2,0.0,2.0
1,2,20000,800,2,1527,4,1,1,1,2,1.0,3.0
2,2,17000,1000,2,1760,4,1,1,1,2,1.0,3.0
3,2,10000,800,2,526,4,2,1,1,2,1.0,2.0
4,2,7500,850,1,1890,4,2,0,1,2,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4741,2,15000,1000,1,219,3,1,1,2,2,3.0,5.0
4742,3,29000,2000,2,1214,3,1,1,3,2,1.0,4.0
4743,3,35000,1750,1,724,3,1,1,3,0,3.0,5.0
4744,3,45000,1500,1,590,3,1,2,2,0,23.0,34.0


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Train Test Split </b></div>

In [20]:
y = df['Rent']
X = df.drop(columns=['Rent'])
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, shuffle=True, test_size=0.3)

In [21]:
print(f'X_train size: {X_train.shape}')

X_train size: (2958, 11)


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Scaling the data </b></div>

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled= pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled= pd.DataFrame(X_test_scaled, columns=X_test.columns)
X_train.head()

,BHK,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors
177,2,600,1,1404,4,2,1,1,2,1.0,2.0
423,2,550,1,1762,4,2,1,1,0,2.0,3.0
2002,2,800,2,1324,0,0,1,2,2,2.0,3.0
4093,3,1675,2,590,3,2,1,3,2,5.0,12.0
2650,1,400,1,1207,2,2,0,1,2,2.0,4.0


In [23]:
X_train_scaled.head()

,BHK,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors
0,0.056772,-0.560958,-1.143388,0.470287,1.023278,1.096230,0.151256,-1.147986,0.571755,-0.435039,-0.552052
1,0.056772,-0.664961,-1.143388,1.036290,1.023278,1.096230,0.151256,-1.147986,-1.749798,-0.156554,-0.388721
2,0.056772,-0.144943,0.872191,0.343806,-1.329049,-1.889634,0.151256,0.278797,0.571755,-0.156554,-0.388721
3,1.411052,1.675122,0.872191,-0.816657,0.435196,1.096230,0.151256,1.705579,0.571755,0.678900,1.081253
4,-1.297509,-0.976972,-1.143388,0.158828,-0.152885,1.096230,-1.828458,-1.147986,0.571755,-0.156554,-0.225391


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Feature Important using Random forest to cross validate </b></div>

In [77]:
# Let's look at the feature importance:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
fig = go.Figure(go.Bar(
            x=rf.feature_importances_,
            y=X_train.columns,
            orientation='h', marker_color='steelblue'))
fig.update_layout(template='plotly_dark', title='<b>Estimating feature importance through the Random Forest model', title_x=0.5, 
                 xaxis_title="Feature importance", yaxis_title='Feature')

fig.show()

#### Observation: As predicted before Area Type is the least important so let us simply drop it

In [24]:
columns_to_drop = ['Area Type']
X_train = X_train.drop(columns=columns_to_drop)
X_train_scaled = X_train_scaled.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)
X_test_scaled = X_test_scaled.drop(columns=columns_to_drop)
#X_test_scaled

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Model selection: Trying out different Regression models on default hyperparameters </b></div>

In [79]:
import lightgbm as lgb
from catboost import CatBoostRegressor
df_models = pd.DataFrame(data=None, columns=['Algorithm', 'r2_train', 'r2_test'])

def make_model(X_tr, X_te, y_tr, y_te, model, model_name: str): 
    model.fit(X_tr, y_tr)
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)
    r2_train = r2_score(y_tr, y_pred_train)
    r2_test = r2_score(y_te, y_pred_test)
    df_models.loc[len(df_models.index)] = [model_name, r2_train, r2_test]
                       

make_model(X_train_scaled, X_test_scaled, y_train, y_test, LinearRegression(), 'LinearRegression')
make_model(X_train_scaled, X_test_scaled, y_train, y_test, Ridge(), 'Ridge')
make_model(X_train_scaled, X_test_scaled, y_train, y_test, Lasso(), 'Lasso')
make_model(X_train_scaled, X_test_scaled, y_train, y_test, ElasticNet(), 'ElasticNet')
make_model(X_train, X_test, y_train, y_test, GradientBoostingRegressor(), 'GradientBoosting')
make_model(X_train, X_test, y_train, y_test, RandomForestRegressor(), 'RandomForest')
make_model(X_train, X_test, y_train, y_test, XGBRegressor(), 'XGBoost')
make_model(X_train, X_test, y_train, y_test, lgb.LGBMRegressor(), 'LightGBM')
make_model(X_train, X_test, y_train, y_test, CatBoostRegressor(), 'CatBoost')

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000995 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 586
[LightGBM] [Info] Number of data points in the train set: 2958, number of used features: 10
[LightGBM] [Info] Start training from score 19244.111562
Learning rate set to 0.048596
0:	learn: 13250.2007679	total: 1.69ms	remaining: 1.69s
1:	learn: 12859.7365992	total: 3.27ms	remaining: 1.63s
2:	learn: 12492.9439272	total: 4.74ms	remaining: 1.57s
3:	learn: 12159.3315760	total: 5.98ms	remaining: 1.49s
4:	learn: 11842.6200999	total: 7.4ms	remaining: 1.47s
5:	learn: 11555.3664108	total: 8.89ms	remaining: 1.47s
6:	learn: 11262.0032765	total: 10.2ms	remaining: 1.45s
7:	learn: 11008.3647306	total: 11.6ms	remaining: 1.44s
8:	learn: 10789.1448878	total: 12.9ms	remaining: 1.42s
9:	learn: 10574.3995605	total: 14.3ms	remaining: 1.4

In [80]:
fig = go.Figure(data=[
    go.Bar(name='r2_train', x=df_models.Algorithm, y=df_models.r2_train),
    go.Bar(name='r2_test', x=df_models.Algorithm, y=df_models.r2_test)
])
fig.update_layout(template='plotly_dark', title='R2 for train and test', title_x=0.5)

#### We can see that Ensemble models working better than linear models which is quite usual as ensemble models mostly outperforms than linear models

#### In this case we will move forawrd with XGBoost as it is performing the best way possible

In [81]:
#import xgboost as xgb
#from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.metrics import r2_score

#best_model = XGBRegressor(reg_lambda=best_lambda,alpha=best_alpha,colsample_bytree=colsample_bytree,subsample=subsample,learning_rate=learning_rate,max_depth=max_depth,random_state=random_state,min_child_weight=min_child_weight,n_estimators=n_estimators)

#best_model.fit(X_train, y_train)

#eval_set = [(X_train, y_train), (X_test, y_test)]
#best_model.fit(X_train, y_train, eval_set=eval_set, early_stopping_rounds=100, verbose=False)

#y_pred_train = best_model.predict(X_train)
#y_pred_test = best_model.predict(X_test)

#train_r2 = r2_score(y_train, y_pred_train)
#test_r2 = r2_score(y_test, y_pred_test)

#print(f"Training R2 Score: {train_r2:.4f}")
#print(f"Testing R2 Score: {test_r2:.4f}")


<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Selecting best hyperparameter using Optuna </b></div>

In [25]:
from xgboost import XGBRegressor
import optuna
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

def objective(trial):
    param = {
        #'tree_method':'gpu_hist',
        'lambda': trial.suggest_loguniform('lambda', 1e-3, 10.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-3, 10.0),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
        'subsample': trial.suggest_categorical('subsample', [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.008, 0.01, 0.012, 0.014, 0.016, 0.018, 0.02]),
        'n_estimators': trial.suggest_int('n_estimators', 10, 1000),
        'max_depth': trial.suggest_categorical('max_depth', [5, 7, 9, 11, 13, 15, 17]),
        'random_state': trial.suggest_categorical('random_state', [2020]),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
    }
    model = XGBRegressor(**param)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=100, verbose=False)
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

In [26]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

[I 2024-07-14 08:39:16,191] A new study created in memory with name: no-name-5c3c1a53-d377-4230-ac49-25a6f3d8eebc
[I 2024-07-14 08:39:16,763] Trial 0 finished with value: 7677.472843079311 and parameters: {'lambda': 0.2926646056141708, 'alpha': 0.04131333995710903, 'colsample_bytree': 1.0, 'subsample': 0.4, 'learning_rate': 0.02, 'n_estimators': 546, 'max_depth': 11, 'random_state': 2020, 'min_child_weight': 150}. Best is trial 0 with value: 7677.472843079311.
[I 2024-07-14 08:39:17,211] Trial 1 finished with value: 7469.479762250803 and parameters: {'lambda': 0.6435216007603007, 'alpha': 6.352495616232941, 'colsample_bytree': 0.8, 'subsample': 0.7, 'learning_rate': 0.018, 'n_estimators': 309, 'max_depth': 11, 'random_state': 2020, 'min_child_weight': 119}. Best is trial 1 with value: 7469.479762250803.
[I 2024-07-14 08:39:17,301] Trial 2 finished with value: 11000.225298633413 and parameters: {'lambda': 0.01271080304277508, 'alpha': 0.04777266222136708, 'colsample_bytree': 0.5, 'subsa

In [27]:
print('Best parameters:', study.best_params)
print('Best RMSE:', study.best_value)

Best parameters: {'lambda': 0.5486948817066843, 'alpha': 2.5389310538514476, 'colsample_bytree': 0.9, 'subsample': 0.7, 'learning_rate': 0.018, 'n_estimators': 944, 'max_depth': 7, 'random_state': 2020, 'min_child_weight': 2}
Best RMSE: 6790.751631126518


In [28]:
#plotting the optimization history of the study
optuna.visualization.plot_optimization_history(study)

In [29]:
#studying the relationship between each value of hyperparameter through this plot
optuna.visualization.plot_parallel_coordinate(study)

In [30]:
# now we plot the accuracy of each hyperparameter for each trial
optuna.visualization.plot_slice(study)

In [31]:
#plotting the accuracy surface for the hyperparameter involved in the xgb
optuna.visualization.plot_contour(study, params=['alpha',
                            'lambda',
                            'subsample',
                            'learning_rate','subsample','n_estimators'])

In [32]:
#Now we look at the role of each parameter in this process
optuna.visualization.plot_param_importances(study)

In [33]:
best_params = study.best_params
best_score = study.best_value
print(f"Best Hyperparameters: {best_params}")
print(f"Best Accuracy: {best_score:.3f}")

Best Hyperparameters: {'lambda': 0.5486948817066843, 'alpha': 2.5389310538514476, 'colsample_bytree': 0.9, 'subsample': 0.7, 'learning_rate': 0.018, 'n_estimators': 944, 'max_depth': 7, 'random_state': 2020, 'min_child_weight': 2}
Best Accuracy: 6790.752


In [34]:
#fetching the best values
best_lambda = best_params['lambda']
best_alpha = best_params['alpha']
colsample_bytree = best_params['colsample_bytree']
subsample = best_params['subsample']
learning_rate = best_params['learning_rate']
max_depth = best_params['max_depth']
random_state = best_params['random_state']
min_child_weight = best_params['min_child_weight']
n_estimators=best_params['n_estimators']


best_model = XGBRegressor(
    reg_lambda=best_lambda,
    alpha=best_alpha,
    colsample_bytree=colsample_bytree,
    subsample=subsample,
    learning_rate=learning_rate,
    max_depth=max_depth,
    random_state=random_state,
    min_child_weight=min_child_weight,
    n_estimators=n_estimators
)

# Fitting the model on the training data
best_model.fit(X_train, y_train)

XGBRegressor(alpha=2.5389310538514476, base_score=None, booster=None,
             callbacks=None, colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.018, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=2, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=944, n_jobs=None,
             num_parallel_tree=None, ...)

In [35]:
print(f'Best model result in Test: {r2_score(y_test, best_model.predict(X_test))}')
print(f'Best model result in Train: {r2_score(y_train, best_model.predict(X_train))}')

Best model result in Test: 0.7710212133830716
Best model result in Train: 0.973815412313862


Using optuna we can clearly see it is overfitting so let us try with Bagging models and fingers crossed lets see if it works out

<a id="1"></a> 
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#0f7a7a; overflow:hidden"><b>Bagging Ensemble</b></div>

In [36]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor


n_samples = X_train_scaled.shape[0]
n_features = X_train_scaled.shape[1]

params = {'base_estimator': [None, LinearRegression(), KNeighborsRegressor(), DecisionTreeRegressor(),GradientBoostingRegressor()],
          'n_estimators': [20,50,100],
          'max_samples': [0.5,1.0],
          'max_features': [0.5,1.0],
          'bootstrap': [True, False],
          'bootstrap_features': [True, False]}

bagging_regressor_grid = GridSearchCV(BaggingRegressor(random_state=1, n_jobs=-1), param_grid=params, cv=3, n_jobs=-1, verbose=1)
bagging_regressor_grid.fit(X_train_scaled, y_train)

Fitting 3 folds for each of 240 candidates, totalling 720 fits


/opt/conda/lib/python3.10/site-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/ensemble/_base.py:166: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/sklearn/en

GridSearchCV(cv=3, estimator=BaggingRegressor(n_jobs=-1, random_state=1),
             n_jobs=-1,
             param_grid={'base_estimator': [None, LinearRegression(),
                                            KNeighborsRegressor(),
                                            DecisionTreeRegressor(),
                                            GradientBoostingRegressor()],
                         'bootstrap': [True, False],
                         'bootstrap_features': [True, False],
                         'max_features': [0.5, 1.0], 'max_samples': [0.5, 1.0],
                         'n_estimators': [20, 50, 100]},
             verbose=1)

In [37]:
print('Train R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_train_scaled, y_train))
print('Test R^2 Score : %.3f'%bagging_regressor_grid.best_estimator_.score(X_test_scaled, y_test))
print('Best R^2 Score Through Grid Search : %.3f'%bagging_regressor_grid.best_score_)
print('Best Parameters : ',bagging_regressor_grid.best_params_)

Train R^2 Score : 0.789
Test R^2 Score : 0.749
Best R^2 Score Through Grid Search : 0.735
Best Parameters :  {'base_estimator': GradientBoostingRegressor(), 'bootstrap': False, 'bootstrap_features': False, 'max_features': 1.0, 'max_samples': 0.5, 'n_estimators': 50}


In [38]:
from sklearn.ensemble import BaggingRegressor
bag_regressor = BaggingRegressor(
    random_state=1,
    n_jobs=-1,
    base_estimator=GradientBoostingRegressor(),
    bootstrap=False,
    bootstrap_features=False,
    max_features=1.0,
    max_samples=0.5,
    n_estimators=50
)
bag_regressor.fit(X_train_scaled, y_train)
Y_preds = bag_regressor.predict(X_test_scaled)
print('Training Coefficient of R^2 : %.3f' % bag_regressor.score(X_train_scaled, y_train))
print('Test Coefficient of R^2 : %.3f' % bag_regressor.score(X_test_scaled, y_test))

Training Coefficient of R^2 : 0.789
Test Coefficient of R^2 : 0.749


#### Ah finally a descent result :) so thats all for this notebook :)